In [1]:
import os
import scipy.io
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split

In [2]:
transform = transforms.Compose([
    # Random augmentation
    # transforms.RandomHorizontalFlip(p=0.15),
    # transforms.RandomRotation(degrees=10),
    # transforms.ColorJitter(brightness=0.2),
    
    # Standard preprocess
    transforms.Resize(128),
    transforms.CenterCrop(112),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [3]:
class OxfordFlowerDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.image_dir = os.path.join(root_dir, "jpg")
        self.transform = transform

        self.labels_path = os.path.join(root_dir, "imagelabels")
        self.labels = scipy.io.loadmat(self.labels_path)["labels"][0] # array shape(1, 8189) -> (8189,)

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        image_file = f"image_{idx+1:05d}.jpg"
        image_path = os.path.join(self.image_dir, image_file)

        image = Image.open(image_path)
        label = self.labels[idx] - 1

        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
dataset = OxfordFlowerDataset("./data", transform)

train_size = int(0.7 * len(dataset))
test_size = len(dataset) - train_size 

train_set, test_set = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

In [5]:
class FlowerClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1), # [3,112,112] -> [16,112,112]
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2), # [16,112,112] -> [16,56,56]

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.block2 = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 14 * 14, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 102)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return x
    

In [6]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using {device}")

model = FlowerClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Using mps


In [7]:
def train_epoch(model, train_loader, loss_function, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, targets)
        loss.backward()
        optimizer.step()

        # Track progress
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()

        if batch_idx % 32 == 0 and batch_idx > 0:
            accuracy = 100 * correct / total
            print(f"Loss: {running_loss / total:.3f} | Accuracy: {accuracy:.1f}")
            running_loss = 0

def evaluate(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)

    return 100 * correct / total

In [8]:
num_epochs = 25
for epoch in range(num_epochs):
    print(f"Epoch {epoch}")
    train_epoch(model, train_loader, loss_function, optimizer, device)
    accuracy = evaluate(model, test_loader, device)
    print(f"Accuracy: {accuracy}")

Epoch 0
Loss: 0.143 | Accuracy: 1.7
Loss: 0.069 | Accuracy: 3.4
Loss: 0.045 | Accuracy: 4.3
Loss: 0.032 | Accuracy: 4.9
Loss: 0.025 | Accuracy: 5.6
Accuracy: 13.34961334961335
Epoch 1
Loss: 0.119 | Accuracy: 10.7
Loss: 0.058 | Accuracy: 11.0
Loss: 0.038 | Accuracy: 11.7
Loss: 0.027 | Accuracy: 12.2
Loss: 0.022 | Accuracy: 12.8
Accuracy: 18.844118844118846
Epoch 2
Loss: 0.107 | Accuracy: 15.2
Loss: 0.051 | Accuracy: 16.4
Loss: 0.035 | Accuracy: 16.6
Loss: 0.025 | Accuracy: 17.3
Loss: 0.020 | Accuracy: 17.8
Accuracy: 24.827024827024825
Epoch 3
Loss: 0.098 | Accuracy: 21.5
Loss: 0.049 | Accuracy: 21.2
Loss: 0.032 | Accuracy: 21.5
Loss: 0.023 | Accuracy: 21.9
Loss: 0.018 | Accuracy: 22.2
Accuracy: 27.96092796092796
Epoch 4
Loss: 0.092 | Accuracy: 24.8
Loss: 0.044 | Accuracy: 25.1
Loss: 0.029 | Accuracy: 25.2
Loss: 0.022 | Accuracy: 25.7
Loss: 0.018 | Accuracy: 25.5
Accuracy: 33.57753357753358
Epoch 5
Loss: 0.081 | Accuracy: 30.8
Loss: 0.040 | Accuracy: 31.5
Loss: 0.027 | Accuracy: 31.0
Los